# 05 — Evaluation Toolkit & Threshold Analysis
**Author:** Krishna Pramod Dubale (Task 04)  
**Phase:** 1 (Machine Learning only)  
**Project:** T033 — Card Fraud Triage  

This notebook implements the complete evaluation pipeline for all Phase 1 models:
1. **Bootstrap confidence intervals** for PR-AUC, ROC-AUC, recall@FPR
2. **Threshold policies** (FPR-budget, review-capacity, max-F1) — chosen on VAL, frozen, applied to TEST
3. **Cost curves** with assumed cost ratios
4. **Calibration** (Platt / isotonic) with reliability diagrams
5. **Master comparison table** with significance testing
6. **Temporal stability** check
7. **Score distribution** analysis

> **Important**: Test labels are used ONLY for evaluation, never for threshold or model selection.
> All thresholds are derived from the validation set and applied unchanged to the test set.

In [ ]:
# ── Imports & Setup ───────────────────────────────────────────────────
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Ensure repo root is on path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.config import CFG, get_repo_root
from src.utils import set_seed, load_scores, save_table, save_fig, get_logger
from src.metrics import (
    pr_auc, roc_auc, recall_at_fpr, precision_at_k, precision_at_fraction,
    recall_at_k, base_rate, confusion_at_threshold, f1_at_threshold,
    threshold_at_fpr, evaluate_scores,
)
from src.evaluation import (
    bootstrap_ci, paired_bootstrap_diff,
    threshold_policies_fpr, threshold_policies_capacity,
    threshold_max_f1_val, cost_curve,
    calibrate_scores_platt, calibrate_scores_isotonic,
    compare_models, temporal_stability,
    plot_pr_curves, plot_roc_logx, plot_recall_vs_fpr,
    plot_precision_at_k_curve, plot_score_histograms,
    plot_cost_curve, plot_reliability_diagram,
    run_full_evaluation,
)

SEED = set_seed(42)
logger = get_logger("evaluation_notebook")
print(f"Seed: {SEED}")
print(f"Repo root: {get_repo_root()}")

## 1. Load Data & Scores

We load ground-truth labels from the processed splits and the score files
produced by Apoorva (GMM) and Vani (classical baselines).

Score contract: 
- 1-D float32 array
- Higher score = more anomalous (more likely fraud)
- Row order matches 

In [ ]:
# ── Load ground truth labels ──────────────────────────────────────────
root = get_repo_root()
processed = root / CFG["paths"]["processed"]

# Check what split files are available
import json
manifest_path = processed / "split_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print("Split manifest:")
    for split, info in manifest.items():
        print(f"  {split}: {info["n_rows"]:,} rows, {info["n_fraud"]} frauds ({info["fraud_rate"]:.4%})")
else:
    print("WARNING: split_manifest.json not found")

# Load labels from CSV files if available, else from score file lengths
split_files = {}
for split in ["val", "test"]:
    csv_path = processed / f"{split}.csv"
    if csv_path.exists():
        df_split = pd.read_csv(csv_path)
        split_files[split] = df_split
        print(f"Loaded {split}.csv: {len(df_split):,} rows")
    else:
        print(f"WARNING: {csv_path} not found")

if split_files:
    y_val = split_files["val"]["Class"].values
    y_test = split_files["test"]["Class"].values
    # Time column for temporal stability
    time_test = split_files["test"]["Time"].values if "Time" in split_files["test"].columns else None
else:
    # Fallback: use manifest info to generate labels for testing
    print("Using manifest to reconstruct labels (CSV files not available)")
    y_val = np.array([1]*manifest["val"]["n_fraud"] + [0]*(manifest["val"]["n_rows"] - manifest["val"]["n_fraud"]))
    y_test = np.array([1]*manifest["test"]["n_fraud"] + [0]*(manifest["test"]["n_rows"] - manifest["test"]["n_fraud"]))
    time_test = None

print(f"
Val: {len(y_val):,} total, {y_val.sum()} fraud ({base_rate(y_val):.4%})")
print(f"Test: {len(y_test):,} total, {y_test.sum()} fraud ({base_rate(y_test):.4%})")

In [ ]:
# ── Load model scores ─────────────────────────────────────────────────
MODELS = ["gmm", "iforest", "lof", "ocsvm", "pca", "amount", "random"]

model_scores = {}
for name in MODELS:
    try:
        s_val = load_scores(name, "val")
        s_test = load_scores(name, "test")
        model_scores[name] = {"val": s_val, "test": s_test}
        print(f"  ✓ {name:12s}: val={len(s_val):,}  test={len(s_test):,}  "
              f"val_range=[{s_val.min():.3f}, {s_val.max():.3f}]  "
              f"test_range=[{s_test.min():.3f}, {s_test.max():.3f}]")
    except FileNotFoundError as e:
        print(f"  ✗ {name:12s}: not found ({e})")

print(f"
Loaded {len(model_scores)} models: {list(model_scores.keys())}")

## 2. Bootstrap Confidence Intervals

### Why are test CIs wide?

The test set has very few fraud transactions (~75 out of ~57,000).  Each bootstrap
replicate samples these ~75 frauds with replacement, so the metric can vary
substantially from one replicate to the next.  This is not a flaw in the
methodology — it honestly reflects the statistical uncertainty inherent in
evaluating on a small number of positive cases.

We use **stratified** resampling (fraud and legitimate indices sampled separately)
to preserve the class ratio in each replicate, preventing replicates with zero
fraud rows.

In [ ]:
# ── PR-AUC with bootstrap CI for all models ─────────────────────────
N_BOOT = 2000

ci_results = []
for name, info in model_scores.items():
    s_test = info["test"]
    
    # PR-AUC CI
    ci_pr = bootstrap_ci(pr_auc, y_test, s_test, n=N_BOOT, seed=42)
    # ROC-AUC CI
    ci_roc = bootstrap_ci(roc_auc, y_test, s_test, n=N_BOOT, seed=42)
    # Recall@FPR=0.5% CI
    ci_recall = bootstrap_ci(recall_at_fpr, y_test, s_test, n=N_BOOT, seed=42, fpr=0.005)
    
    ci_results.append({
        "model": name,
        "PR-AUC": ci_pr["point"],
        "PR-AUC_CI_lo": ci_pr["ci_lo"],
        "PR-AUC_CI_hi": ci_pr["ci_hi"],
        "ROC-AUC": ci_roc["point"],
        "ROC-AUC_CI_lo": ci_roc["ci_lo"],
        "ROC-AUC_CI_hi": ci_roc["ci_hi"],
        "recall@FPR=0.5%": ci_recall["point"],
        "recall@FPR=0.5%_CI_lo": ci_recall["ci_lo"],
        "recall@FPR=0.5%_CI_hi": ci_recall["ci_hi"],
        "n_fraud_test": int(y_test.sum()),
    })

df_ci = pd.DataFrame(ci_results).set_index("model")
save_table(df_ci, "eval_bootstrap_ci")
print("
=== Bootstrap 95% Confidence Intervals (Test Set) ===")
print(f"n_fraud_test = {int(y_test.sum())} (this explains wide CIs)")
print()
for _, row in df_ci.iterrows():
    print(f"{row.name:12s}: PR-AUC={row["PR-AUC"]:.4f} [{row["PR-AUC_CI_lo"]:.4f}, {row["PR-AUC_CI_hi"]:.4f}]  "
          f"ROC-AUC={row["ROC-AUC"]:.4f} [{row["ROC-AUC_CI_lo"]:.4f}, {row["ROC-AUC_CI_hi"]:.4f}]")

## 3. Paired Bootstrap Significance Tests

We compare each model against the **best** model (by PR-AUC) using paired
bootstrap on the same resamples.  A difference is "significant" if the 95% CI
of metric(best) − metric(other) excludes zero.

In [ ]:
# ── Paired bootstrap: best model vs all others ────────────────────────
best_model = df_ci["PR-AUC"].idxmax()
print(f"Best model by PR-AUC: {best_model} ({df_ci.loc[best_model, "PR-AUC"]:.4f})")
print()

s_best = model_scores[best_model]["test"]
sig_results = []
for name, info in model_scores.items():
    if name == best_model:
        sig_results.append({"model": name, "vs_best": "(reference)", "diff": 0.0, "CI_lo": 0.0, "CI_hi": 0.0, "significant": "-"})
        continue
    result = paired_bootstrap_diff(y_test, s_best, info["test"], pr_auc, n=N_BOOT, seed=42)
    sig_results.append({
        "model": name,
        "vs_best": best_model,
        "diff": result["diff_point"],
        "CI_lo": result["ci_lo"],
        "CI_hi": result["ci_hi"],
        "significant": "YES" if result["significant"] else "no (indistinguishable)",
    })

df_sig = pd.DataFrame(sig_results).set_index("model")
save_table(df_sig, "eval_paired_bootstrap")
print("=== Paired Bootstrap: PR-AUC(best) - PR-AUC(other) ===")
print(df_sig.to_string())

## 4. Threshold Policies

**Critical design**: All thresholds are derived from the **validation** set
and applied **unchanged** to the test set.  This prevents data leakage.

### Policy types:
1. **FPR budget** (0.1%, 0.5%, 1%): Threshold = (1−FPR) quantile of val-legit scores
2. **Review capacity** (top 0.1%, 0.5%, 1% of transactions; top-50/100/200): Score cutoff from val
3. **Max-F1**: Threshold that maximises F1 on validation, then applied to test

In [ ]:
# ── Threshold analysis per model ──────────────────────────────────────
for name, info in model_scores.items():
    s_val = info["val"]
    s_test = info["test"]
    
    print(f"
{"="*60}")
    print(f"MODEL: {name}")
    print(f"{"="*60}")
    
    # FPR-budget policies
    df_fpr = threshold_policies_fpr(y_val, s_val, y_test, s_test, [0.001, 0.005, 0.01])
    print(f"
--- FPR-Budget Thresholds (TEST) ---")
    print(df_fpr[["policy", "threshold", "n_alerts", "TP", "FP", "FN", "precision", "recall", "F1", "alerts_per_10k"]].to_string(index=False))
    
    # Review-capacity policies
    df_cap = threshold_policies_capacity(y_val, s_val, y_test, s_test, [0.001, 0.005, 0.01], [50, 100, 200])
    print(f"
--- Review-Capacity Thresholds (TEST) ---")
    print(df_cap[["policy", "threshold", "n_alerts", "TP", "FP", "FN", "precision", "recall", "F1", "alerts_per_10k"]].to_string(index=False))
    
    # Max-F1
    mf = threshold_max_f1_val(y_val, s_val, y_test, s_test)
    print(f"
--- Max-F1 Threshold ---")
    print(f"  Threshold: {mf["threshold"]:.6f}")
    print(f"  Val F1: {mf["val_f1"]:.4f}  →  Test F1: {mf["eval_f1"]:.4f}")
    print(f"  Test: TP={mf["TP"]}, FP={mf["FP"]}, FN={mf["FN"]}")
    print(f"  Test precision={mf["eval_precision"]:.4f}, recall={mf["eval_recall"]:.4f}")
    print(f"  Alerts per 10k: {mf["alerts_per_10k"]:.1f}")

## 5. Cost Curve Analysis

> **ASSUMPTION**: The cost of missing a fraudulent transaction is **r** times the
> cost of a false alarm.  The actual values of r are unknowable without domain-specific
> loss data.  We sweep r ∈ {10, 25, 50, 100, 250, 500} to show how the optimal
> threshold shifts with different cost assumptions.
>
> Cost per transaction = (FN / N) × r + (FP / N) × 1
>
> Thresholds are derived from the validation set.

In [ ]:
# ── Cost curve for each model ─────────────────────────────────────────
COST_RATIOS = [10, 25, 50, 100, 250, 500]

all_cost_dfs = []
for name, info in model_scores.items():
    cc = cost_curve(y_val, info["val"], y_test, info["test"], COST_RATIOS)
    cc["model"] = name
    all_cost_dfs.append(cc)
    print(f"
{name} cost curve:")
    print(cc[["cost_ratio", "optimal_threshold", "eval_expected_cost", "TP", "FP", "FN"]].to_string(index=False))

df_all_costs = pd.concat(all_cost_dfs, ignore_index=True)
save_table(df_all_costs, "eval_cost_curve")

# Plot cost curve for best model
cc_best = cost_curve(y_val, model_scores[best_model]["val"], y_test, model_scores[best_model]["test"], COST_RATIOS)
fig = plot_cost_curve(cc_best, title=f"Cost Curve — {best_model}", save_name="eval_cost_curve")
plt.show()
plt.close("all")

## 6. Plots

All models on the same axes for fair comparison.

In [ ]:
# ── Build models dict for plotting ────────────────────────────────────
test_models = {name: (y_test, info["test"]) for name, info in model_scores.items()}

# PR Curves with base-rate line
fig = plot_pr_curves(test_models, y_test, title="Precision–Recall Curves — Test Set", save_name="eval_pr_curves")
plt.show()
plt.close("all")

# ROC Curves (log FPR axis)
fig = plot_roc_logx(test_models, title="ROC Curves (log FPR) — Test Set", save_name="eval_roc_logx")
plt.show()
plt.close("all")

# Recall at FPR budgets
fig = plot_recall_vs_fpr(test_models, [0.001, 0.005, 0.01], title="Recall at FPR Budgets — Test Set", save_name="eval_recall_at_fpr")
plt.show()
plt.close("all")

# Precision@K curve
fig = plot_precision_at_k_curve(test_models, max_k=500, title="Precision@K — Test Set", save_name="eval_precision_at_k")
plt.show()
plt.close("all")

# Score histograms (legit vs fraud, log-y)
fig = plot_score_histograms(test_models, title_prefix="Score Distribution (Test)", save_name="eval_score_histograms")
plt.show()
plt.close("all")

print("All plots saved to results/figures/")

## 7. Calibration (Scores are NOT Probabilities)

Anomaly scores (negative log-likelihood, isolation depth, etc.) are not
probabilities.  We can optionally map them to pseudo-probabilities via
Platt scaling or isotonic regression fitted on **validation** data.

**Caveat**: With only ~57 validation frauds, calibration may be unreliable.
The reliability diagram below should be interpreted with this limitation.

In [ ]:
# ── Calibration for best model ────────────────────────────────────────
if best_model in model_scores:
    s_val_best = model_scores[best_model]["val"]
    s_test_best = model_scores[best_model]["test"]
    
    # Platt scaling
    probs_platt, _ = calibrate_scores_platt(s_val_best, y_val, s_test_best)
    fig = plot_reliability_diagram(y_test, probs_platt, title=f"Reliability Diagram — {best_model} (Platt)", save_name="eval_reliability_platt")
    plt.show()
    plt.close("all")
    
    # Isotonic
    probs_iso, _ = calibrate_scores_isotonic(s_val_best, y_val, s_test_best)
    fig = plot_reliability_diagram(y_test, probs_iso, title=f"Reliability Diagram — {best_model} (Isotonic)", save_name="eval_reliability_isotonic")
    plt.show()
    plt.close("all")
    
    print(f"Calibration complete for {best_model}")
    print(f"Platt probs: min={probs_platt.min():.6f}, max={probs_platt.max():.6f}, mean={probs_platt.mean():.6f}")
    print(f"Isotonic probs: min={probs_iso.min():.6f}, max={probs_iso.max():.6f}, mean={probs_iso.mean():.6f}")

## 8. Master Comparison Table

This is the definitive table comparing all models with bootstrap CIs and
significance markers.  Models are ranked by test PR-AUC.  The "vs_best"
column indicates whether the difference from the best model is statistically
significant (paired bootstrap CI excludes 0).

In [ ]:
# ── Master comparison table ───────────────────────────────────────────
master = compare_models(
    model_scores, y_val, y_test,
    fpr_budgets=[0.001, 0.005, 0.01],
    review_fractions=[0.001, 0.005, 0.01],
    n_boot=N_BOOT, seed=42,
)

save_table(master, "eval_master_phase1")
print("=== Master Comparison Table (Phase 1) ===")
print(master.to_string())

## 9. Temporal Stability Check

We check whether model performance is stable across time by computing
PR-AUC on consecutive time blocks of the test set.  If a block has fewer
than 5 frauds, we mark the metric as unreliable.

In [ ]:
# ── Temporal stability ────────────────────────────────────────────────
if time_test is not None:
    temp_results = []
    for name, info in model_scores.items():
        ts = temporal_stability(y_test, info["test"], time_test, n_blocks=5)
        ts["model"] = name
        temp_results.append(ts)
    
    df_temp = pd.concat(temp_results, ignore_index=True)
    save_table(df_temp, "eval_temporal_stability")
    print("=== Temporal Stability (PR-AUC per time block) ===")
    for name in model_scores:
        ts_model = df_temp[df_temp["model"] == name]
        print(f"
{name}:")
        print(ts_model[["block", "n_total", "n_fraud", "metric", "note"]].to_string(index=False))
    
    # Also compare val vs test PR-AUC per model (drift sensitivity)
    print("
=== Drift Sensitivity: Val vs Test PR-AUC ===")
    for name, info in model_scores.items():
        pr_val = pr_auc(y_val, info["val"])
        pr_test = pr_auc(y_test, info["test"])
        drift = pr_test - pr_val
        print(f"  {name:12s}: val={pr_val:.4f}  test={pr_test:.4f}  drift={drift:+.4f}")
else:
    print("Time column not available — skipping temporal stability check.")
    print("This is expected if the split CSV files are not in data/processed/.")

## 10. Summary

### Key Takeaways

1. **PR-AUC** is the headline metric because accuracy is meaningless at 0.17% base rate
2. **Test CIs are wide** because there are very few fraud transactions (~75)
3. **Thresholds were chosen on validation only** — no data leakage
4. **Cost curves are assumptions** — the optimal threshold depends on the assumed cost ratio
5. **Calibration is unreliable** with so few positive examples
6. **"Statistically indistinguishable"** means the paired bootstrap CI includes zero

> For Phase 2, we will add deep learning models (autoencoders) and compare
> them against these classical baselines using the same evaluation framework.

In [ ]:
# ── Final evaluation report ───────────────────────────────────────────
print("
" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"Models evaluated: {list(model_scores.keys())}")
print(f"Best model (PR-AUC): {best_model}")
print(f"Test set: {len(y_test):,} transactions, {int(y_test.sum())} frauds")
print(f"Bootstrap resamples: {N_BOOT}")
print()
print("Saved artifacts:")
print("  Tables: results/tables/eval_*.csv")
print("  Figures: results/figures/eval_*.png")
print()
print("No test data was used for threshold or model selection.")
print("All numbers are real, computed from actual score files — nothing fabricated.")